# 📊 AI SQL Analyst Agent

**An end-to-end, AI-powered data analysis pipeline that allows users to query relational databases using natural language. This project leverages a locally-hosted Large Language Model (Mistral-Nemo-Instruct) to dynamically introspect database schemas, write and execute secure SQL queries, and generate automated business insights alongside Python-based data visualizations.The system is built on a decoupled architecture, separating the heavy LLM inference and database execution (FastAPI) from the user-facing interface (Streamlit).**


**✨ System Architecture & Workflow**
* Dynamic Database Initialization: Automatically ingests flat CSV files from Kaggle datasets and maps them into a fully relational SQLite database.

* Schema Introspection: Extracts table structures, column names, and data types dynamically to build context for the LLM.

* NL-to-SQL Generation: LangChain prompts a 4-bit quantized Mistral LLM to translate user questions into syntactically correct SQLite queries.

* Execution & Error Handling: The backend executes the SQL query using Pandas, catching and returning any engine-level errors safely.

* Insights & Visualization: The resulting dataset is fed back into the LLM, which is constrained via a Structured Output Parser to return a strictly formatted JSON object containing a business summary and executable Matplotlib code.

**🛠️ Technology Stack**
* Large Language Model: Mistral-Nemo-Instruct-2407

* Model Optimization: BitsAndBytes (4-bit Quantization), Accelerate, PyTorch

* AI Orchestration: LangChain (Custom LLM Wrappers, PromptTemplates, StructuredOutputParsers)

* Backend: FastAPI, Uvicorn, SQLite, Pandas

* Frontend: Streamlit, Matplotlib, Pyngrok

In [ ]:
!pip -q install streamlit pyngrok fastapi uvicorn requests
!pip -q install "transformers>=4.43" "accelerate>=0.33" "bitsandbytes>=0.43" "autoawq>=0.2.7" torch --extra-index-url https://download.pytorch.org/whl/cu121
!pip -q install langchain langchain-community langchain-core langchain-classic pandas matplotlib

In [ ]:
%%writefile app.py
import os
import re
import json
import sqlite3
import pandas as pd
import torch
import traceback
import matplotlib.pyplot as plt
import streamlit as st
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from langchain_core.language_models.llms import LLM
from langchain_core.prompts import PromptTemplate
from langchain_classic.chains import LLMChain
from typing import Any

st.set_page_config(page_title="AI SQL Analyst", layout="wide")

@st.cache_resource(show_spinner="Loading AI Model into GPU...")
def load_model():
    base_id = "mistralai/Mistral-Nemo-Instruct-2407"
    bnb_cfg = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16)
    tokenizer = AutoTokenizer.from_pretrained(base_id, use_fast=True)
    model = AutoModelForCausalLM.from_pretrained(base_id, quantization_config=bnb_cfg, device_map="auto")
    return tokenizer, model

tokenizer, model = load_model()

class CustomHFLLM(LLM):
    def _call(self, prompt: str, stop: Any = None) -> str:
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        outputs = model.generate(
            **inputs, 
            max_new_tokens=600, 
            do_sample=True, 
            temperature=0.1, 
            top_p=0.95, 
            pad_token_id=tokenizer.eos_token_id
        )
        return tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()

    @property
    def _llm_type(self) -> str:
        return "custom_huggingface_quantized"

llm = CustomHFLLM()

def extract_block(text, block_type="sql"):
    text = str(text)
    pattern = rf"```(?:{block_type})?\s*(.*?)\s*```"
    matches = re.findall(pattern, text, re.DOTALL | re.IGNORECASE)
    if matches:
        return matches[-1].strip()
    if f"```{block_type}" in text:
        return text.split(f"```{block_type}")[-1].replace("```", "").strip()
    return text.strip()

st.title("📊 AI SQL Analyst")
st.sidebar.header("Data Source")

uploaded_file = st.sidebar.file_uploader("Upload a CSV dataset", type=["csv"])
db_path = "analytics.db"
conn = sqlite3.connect(db_path, check_same_thread=False)

if uploaded_file is not None:
    raw_name = uploaded_file.name.replace('.csv', '').replace(' ', '_')
    table_name = re.sub(r'\W+', '', raw_name).lower()
    
    df_uploaded = pd.read_csv(uploaded_file)
    df_uploaded.columns = [re.sub(r'\W+', '', c.strip().replace(' ', '_')) for c in df_uploaded.columns]
    df_uploaded.to_sql(table_name, conn, if_exists='replace', index=False)
    
    st.sidebar.success(f"Loaded `{table_name}` ({len(df_uploaded)} rows)")
    
    cursor = conn.cursor()
    cursor.execute(f"PRAGMA table_info({table_name});")
    columns = [f"{col[1]} ({col[2]})" for col in cursor.fetchall()]
    schema = f"Table: {table_name}\nColumns: {', '.join(columns)}"
    st.sidebar.code(schema)
else:
    st.info("👈 Upload a CSV file in the sidebar to begin.")
    st.stop()

user_query = st.chat_input("Ask a question about your data...")

if user_query:
    st.chat_message("user").write(user_query)
    
    with st.chat_message("assistant"):
        with st.spinner("Generating SQL..."):
            sql_template = (
                "<s>[INST] You are an expert SQL Data Analyst.\n"
                "Write a valid SQLite query to answer the question using the database schema below.\n"
                "Output ONLY the SQL code inside a ```sql ``` code block. No explanations.\n\n"
                f"Schema:\n{schema}\n\n"
                f"Question: {user_query} [/INST]\n```sql"
            )
            raw_sql = LLMChain(llm=llm, prompt=PromptTemplate(template=sql_template, input_variables=[])).run({})
            
            if not raw_sql.strip().startswith("```sql") and "```sql" not in raw_sql:
                raw_sql = "```sql\n" + raw_sql
            sql_query = extract_block(raw_sql, "sql")
            
            st.markdown("**Generated SQL:**")
            st.code(sql_query, language="sql")
            
        with st.spinner("Executing Query..."):
            try:
                df_result = pd.read_sql_query(sql_query, conn)
            except Exception as e:
                st.error(f"SQL Execution Error: {e}")
                st.stop()
                
            if df_result.empty:
                st.warning("Query returned 0 rows.")
                st.stop()
                
            st.markdown("**Query Results:**")
            st.dataframe(df_result)
            
        with st.spinner("Analyzing & Plotting..."):
            data_str = df_result.head(10).to_string(index=False)
            columns_str = str(list(df_result.columns))
            
            analysis_template = (
                "<s>[INST] You are an expert Data Analyst. Analyze this query result and write Python visualization code using matplotlib.\n\n"
                f"User Question: {user_query}\n"
                f"Actual DataFrame Columns: {columns_str}\n"
                f"Data:\n{data_str}\n\n"
                "CRITICAL RULES FOR viz_code:\n"
                "1. `df_result` is already loaded. Do NOT use pd.read_csv.\n"
                "2. MUST use the exact columns listed above, or positional indexing (df_result.iloc[:,0]).\n"
                "3. If df_result has 1 row/column, plot a simple bar.\n\n"
                "Respond strictly in the following JSON format:\n"
                "```json\n"
                "{{\n"
                '  "analysis": "1-2 sentences of key business takeaways.",\n'
                '  "viz_code": "import matplotlib.pyplot as plt\\nfig, ax = plt.subplots()\\nax.bar(df_result.columns[0], df_result.iloc[0,0])"\n'
                "}}\n"
                "```\n"
                "Ensure double quotes in code are escaped. [/INST]\n```json"
            )
            raw_response = LLMChain(llm=llm, prompt=PromptTemplate(template=analysis_template, input_variables=[])).run({})
            
            if not raw_response.strip().startswith("```json") and "```json" not in raw_response:
                full_json_str = "```json\n" + raw_response + "\n```"
            else:
                full_json_str = raw_response
                
            json_block = extract_block(full_json_str, "json")
            
            try:
                parsed = json.loads(json_block, strict=False)
                analysis_text = parsed.get("analysis", "")
                viz_code = parsed.get("viz_code", "")
            except:
                analysis_match = re.search(r'"analysis"\s*:\s*"([^"]+)"', json_block)
                viz_match = re.search(r'"viz_code"\s*:\s*"(.*?)"\s*\}', json_block, re.DOTALL)
                analysis_text = analysis_match.group(1) if analysis_match else "Could not parse analysis."
                viz_code = viz_match.group(1).encode().decode('unicode_escape') if viz_match else ""

            st.markdown("**Business Analysis:**")
            st.info(analysis_text)
            
            if viz_code:
                st.markdown("**Visualization:**")
                with st.expander("View Code"):
                    st.code(viz_code, language="python")
                try:
                    fig, ax = plt.subplots()
                    exec(viz_code, globals(), {"df_result": df_result, "plt": plt, "fig": fig, "ax": ax})
                    st.pyplot(plt.gcf())
                    plt.clf()
                except Exception as e:
                    st.error(f"Render Error: {e}")

In [ ]:
import os
import subprocess
import time
from pyngrok import ngrok
from getpass import getpass

# Kill previous sessions
os.system("pkill -f streamlit")
ngrok.kill()

# Securely request ngrok token
print("Enter your ngrok authtoken (get it from https://dashboard.ngrok.com/auth):")
authtoken = getpass()
ngrok.set_auth_token(authtoken)

# Start Streamlit
process = subprocess.Popen(["streamlit", "run", "app.py", "--server.port", "8501", "--server.headless", "true"])
time.sleep(4)

# Open tunnel
public_url = ngrok.connect(8501)
print(f"\n✅ App is live! Click here: {public_url.public_url}")